# 03 — Construire une interface personnalisée

Ce notebook reconstruit, brique par brique, l'interface multi-tâches de `examples/09_custom_multitask_pdf.py`.

Idée directrice : **une interface Kili n'est qu'un arbre JSON de jobs**. On peut en combiner autant qu'on veut, de types différents, sur un même asset.

In [ ]:
import json

from kili_examples.interfaces import (
    build_category,
    build_classification_job,
    build_json_interface,
    build_ner_job,
    build_object_detection_job,
    build_transcription_job,
)

## Brique 1 — classer le document

Un job `CLASSIFICATION` mono-classe. La catégorie `DECLARATION_SINISTRE` déclare un `children` : elle ouvrira un sous-job.

In [ ]:
job_type = build_classification_job(
    instruction="Quel est le type de ce document ?",
    categories={
        "DECLARATION_SINISTRE": build_category(
            "Déclaration de sinistre",
            children=["SOUS_TYPE_SINISTRE"],
        ),
        "FACTURE": build_category("Facture"),
        "ATTESTATION": build_category("Attestation"),
    },
)
print(json.dumps(job_type, indent=2, ensure_ascii=False))

## Brique 2 — le sous-job conditionnel

`isChild=True` : ce job ne s'affiche que si la catégorie parente est cochée.

In [ ]:
job_sous_type = build_classification_job(
    instruction="Nature du sinistre déclaré ?",
    categories={
        "AUTO": build_category("Automobile"),
        "HABITATION": build_category("Habitation"),
    },
    required=False,
    is_child=True,
)
print(json.dumps(job_sous_type, indent=2, ensure_ascii=False))

## Brique 3 — les entités nommées

`NAMED_ENTITIES_RECOGNITION` : l'annotateur surligne des portions de texte.

In [ ]:
job_entites = build_ner_job(
    instruction="Surlignez les entités clés.",
    categories={
        "NUMERO_CONTRAT": build_category("Numéro de contrat"),
        "DATE_ACCIDENT": build_category("Date de l'accident"),
    },
    required=False,
)
print(json.dumps(job_entites, indent=2, ensure_ascii=False))

## Brique 4 — un champ à transcrire

`TRANSCRIPTION` : pas de catégories, une zone de saisie libre (`input: "textarea"`).

In [ ]:
job_montant = build_transcription_job(
    instruction="Montant total mentionné, en euros.",
    required=False,
)
print(json.dumps(job_montant, indent=2, ensure_ascii=False))

## Brique 5 — des zones à entourer

`OBJECT_DETECTION` avec `tools=["rectangle"]`. Le `mlTask` est le même pour les boîtes, les polygones et les masques : c'est `tools` qui fait la différence.

In [ ]:
job_zones = build_object_detection_job(
    instruction="Entourez les zones à vérifier.",
    categories={
        "SIGNATURE": build_category("Signature"),
        "TABLEAU_MONTANTS": build_category("Tableau des montants"),
    },
    tools=["rectangle"],
    required=False,
)
print(json.dumps(job_zones, indent=2, ensure_ascii=False))

## Assemblage

On réunit les cinq briques dans un seul `json_interface`.

In [ ]:
json_interface = build_json_interface(
    {
        "TYPE_DOCUMENT": job_type,
        "SOUS_TYPE_SINISTRE": job_sous_type,
        "ENTITES_DOCUMENT": job_entites,
        "MONTANT_TOTAL": job_montant,
        "ZONES_CLES": job_zones,
    }
)
print("Jobs :", list(json_interface["jobs"]))

### Vérification : les références de sous-jobs

Une erreur classique est de déclarer `children=["X"]` sans que le job `X` existe, ou sans le marquer `isChild=True`. Kili accepte l'interface mais le sous-job ne s'affiche jamais. Contrôlons.

In [ ]:
for nom, job in json_interface["jobs"].items():
    for cle, cat in job["content"].get("categories", {}).items():
        for enfant in cat.get("children", []):
            existe = enfant in json_interface["jobs"]
            est_enfant = existe and json_interface["jobs"][enfant]["isChild"]
            print(
                f"{nom}.{cle} -> {enfant} : existe={existe}, isChild={est_enfant}"
            )

## Une prédiction pour plusieurs jobs

Le point clé : **un seul** `json_response` adresse tous les jobs. Une clé de premier niveau par job rempli, et chaque valeur suit la forme de son `mlTask`.

In [ ]:
import importlib.util
from pathlib import Path

chemin = Path.cwd().parent / "examples" / "09_custom_multitask_pdf.py"
spec = importlib.util.spec_from_file_location("ex09", chemin)
ex09 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ex09)

reponse = ex09.predict({"external_id": "constat_amiable_001"})
print(json.dumps(reponse, indent=2, ensure_ascii=False))

Récapitulatif des formes de réponse par type de job :

| `mlTask` | Forme de la réponse |
| --- | --- |
| `CLASSIFICATION` | `{"categories": [{"name": ..., "confidence": ...}]}` |
| `TRANSCRIPTION` | `{"text": "..."}` |
| `NAMED_ENTITIES_RECOGNITION` (texte) | `{"annotations": [{"beginOffset", "content", "categories"}]}` |
| `OBJECT_DETECTION` (image) | `{"annotations": [{"boundingPoly", "type", "categories", "mid"}]}` |
| `OBJECT_DETECTION` / NER (PDF) | idem + `polys` et `pageNumberArray` dans une liste `annotations` imbriquée |

## Validation hors ligne

Avant tout envoi, on confronte la réponse à l'interface avec le parseur du SDK.

In [ ]:
from kili.services.label_data_parsing.json_response import ParsedJobs
from kili.services.label_data_parsing.types import Project

parsed = ParsedJobs(
    json_response=reponse,
    project_info=Project(
        jsonInterface=ex09.build_interface()["jobs"],
        inputType="PDF",
    ),
)
print("Réponse cohérente avec l'interface de l'exemple 09.")

## Pour aller plus loin

- `examples/09_custom_multitask_pdf.py` : le script complet et exécutable ;
- `docs/` : une page par type d'annotation, avec le `json_interface` et la charge utile de prédiction correspondante ;
- `docs/incertitudes.md` : les points non vérifiés contre une instance réelle.